In [ ]:
from google.colab import files
files.upload()
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('kaggle datasets download -d emmarex/plantdisease --unzip -p /content/PlantVillage')
print('Dataset ready!')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

In [ ]:
DATASET_PATH = '/content/PlantVillage'
IMG_SIZE = 128
BATCH    = 32

categories  = sorted(os.listdir(DATASET_PATH))
NUM_CLASSES = len(categories)
print('Total Classes:', NUM_CLASSES)
for c in categories:
    print(c, '->', len(os.listdir(os.path.join(DATASET_PATH, c))), 'images')

In [ ]:
gen = ImageDataGenerator(rescale=1./255, rotation_range=20,
                         horizontal_flip=True, validation_split=0.2)

train_data = gen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE,IMG_SIZE),
                                     batch_size=BATCH, class_mode='categorical', subset='training')
val_data   = gen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE,IMG_SIZE),
                                     batch_size=BATCH, class_mode='categorical', subset='validation')

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(train_data, epochs=10, validation_data=val_data, verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
val_loss, val_acc = model.evaluate(val_data)
print(f'Validation Accuracy: {val_acc*100:.2f}%')

In [ ]:
class_labels = list(train_data.class_indices.keys())

def predict_image(img_path):
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = img_to_array(img) / 255.0
    arr = arr.reshape(1, IMG_SIZE, IMG_SIZE, 3)
    pred = class_labels[np.argmax(model.predict(arr))]
    conf = model.predict(arr)[0].max() * 100
    plt.imshow(img); plt.axis('off')
    plt.title(f'Predicted: {pred} ({conf:.1f}%)')
    plt.show()

sample_cls = categories[0]
sample_img = os.listdir(os.path.join(DATASET_PATH, sample_cls))[0]
predict_image(os.path.join(DATASET_PATH, sample_cls, sample_img))